# LightGBM con futuro · diario de experimentos (Pablo · A · `futuro/`)

> ⚠️ Usa información de **otros snapshots de la sesión** (pasado y futuro): rompe a propósito la regla 1 del equipo.
> Por eso vive en `futuro/` y guarda en `futuro/oof|preds|submissions`: el stacking normal de `main` no lo ve; solo
> `stacking/stack.py --futuro` lo combina (ver `futuro/README.md`).

**Qué es:** `futuro/m_lgb_future.py`, un LightGBM LambdaRank (parámetros de referencia de `EQUIPO.md`) que puntúa cada tarea con
cuatro bloques de features de `futuro/features_future.py` (ninguna usa la etiqueta):

| Bloque | Qué contiene |
|---|---|
| `base` | columnas originales del snapshot |
| `logic` | reglas dentro del snapshot: plazo por tipo, factibilidad, ventana típica, EDF… (`logic_*`) |
| `past` | cuánto lleva la tarea en la lista, cambios respecto al snapshot anterior, racha de snapshots (`p_*`, `blk_*`, `gap_*`) |
| `future` | segundos hasta la siguiente instrucción al avión y su tipo, si encaja con la tarea, cuándo desaparece la tarea (`f_*`) |

**Cómo usar este notebook** (igual que `NN_pablo.ipynb`)
1. Ejecuta *Preparación* una vez (calcula las features, ~1–2 min).
2. Cada bloque: **hipótesis** → **variantes** (UNA cosa cambia respecto a la base) → **tabla** → conclusión.
3. Lo que gane se combina en `MEJOR`, se confirma con 3 semillas y se guarda para el stacking.

**Cómo leer las cifras:** `cv` = top-3 out-of-fold con `Data/folds.csv`. Pruebas rápidas con 1 semilla: diferencias < ~0.005
son ruido. `min_atco` = peor controlador (el privado es un solo controlador). Referencias: LightGBM limpio de Adri 0.648,
red con futuro 0.887, LightGBM con futuro antiguo (`CLAUDE.md`, folds viejos) 0.83.

**Qué no se repite de `LGB_adri.ipynb`** (solo se lee, no se toca): Adri ya probó, con datos limpios, sus grupos de features
(`snapshot`, `tipo`, `avion`, `tipo_estado`, `contexto`): sueltos aportan poco (+0.001 a +0.004) y **todos juntos empeoran**
(0.648 frente a 0.651). Eso es trabajo suyo y no se repite aquí. Sus **candidatos de parámetros** (`lento`, `regularizado`,
`xendcg`) están escritos pero no los ha lanzado: se prueban aquí con los datos con futuro (bloque 2), para que los dos
LightGBM se comparen con lo mismo.

## Preparación

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "futuro") else Path.cwd()
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "cv")); sys.path.insert(0, str(ROOT / "futuro"))
import m_lgb_future

data = m_lgb_future.build_data()      # una sola vez; todas las pruebas lo reutilizan
REF = {"LGB limpio (Adri)": 0.648, "red con futuro": 0.887}

In [ ]:
# Configuración base = parámetros de referencia de EQUIPO.md con los 4 bloques de features. Cada experimento cambia UNA clave.
BASE = dict(feats="base,logic,past,future", n_estimators=400, lr=0.05, leaves=31, min_child=50, subsample=0.8,
            colsample=0.7, trunc=5, reg_lambda=0.0, objective="lambdarank")


def probar(nombre, motivo, semillas=1, guardar=False, **cambios):
    """Lanza la CV con BASE + cambios; queda apuntado en experiments/lgb_future_runs.csv con su motivo.
    guardar=True genera futuro/oof/, futuro/preds/ y futuro/submissions/ (solo para la versión que va al stacking)."""
    cfg = {**BASE, **cambios}
    args = ["--name", nombre, "--tag", motivo] + (["--quick"] if not guardar else [])
    for k, v in cfg.items():
        args += [f"--{k.replace('_', '-')}", str(v)]
    a = m_lgb_future.parse(args)
    a.seeds = semillas
    return m_lgb_future.run(a, data)


def lanzar(bloque, variantes, semillas=1, sobre=None):
    """variantes: lista de (nombre, motivo, {cambios}). Siempre incluye la base para comparar en igualdad."""
    sobre = sobre or {}
    res = [("base", "referencia", probar(f"{bloque}/base", "referencia", semillas, **sobre))]
    for nombre, motivo, cambios in variantes:
        res.append((nombre, motivo, probar(f"{bloque}/{nombre}", motivo, semillas, **{**sobre, **cambios})))
    return tabla_bloque(res)


def tabla_bloque(res):
    t = pd.DataFrame(res, columns=["variante", "por qué", "cv"])
    ref = t.loc[t.variante == "base", "cv"].iloc[0]
    t["Δ vs base"] = t.cv - ref
    t["veredicto"] = t["Δ vs base"].map(lambda x: "—" if abs(x) < 1e-9 else ("mejora" if x > 0.005 else
                                                   ("empeora" if x < -0.005 else "ruido")))
    return (t.sort_values("cv", ascending=False).style
            .format({"cv": "{:.4f}", "Δ vs base": "{:+.4f}"})
            .background_gradient(subset=["Δ vs base"], cmap="RdYlGn", vmin=-0.02, vmax=0.02).hide(axis="index"))


def historial(filtro=""):
    """Todo lo probado (también desde la terminal), ordenado por CV."""
    r = pd.read_csv(m_lgb_future.LOG)
    r = r[r.name.str.contains(filtro, regex=False)] if filtro else r
    cols = ["when", "name", "tag", "cv", "min_atco", "seeds", "feats", "n_feats", "leaves", "min_child", "lr",
            "n_estimators", "colsample", "secs"]
    return r[[c for c in cols if c in r]].sort_values("cv", ascending=False).reset_index(drop=True)

## 0 · Punto de partida

Base con 3 semillas, generada desde la terminal (`python futuro/m_lgb_future.py`): es la versión `lgb_future` que ya está en
`futuro/oof/` y `futuro/preds/` para Juanma. La celda la mide con la métrica del equipo (`cv/metric.py`).

In [ ]:
from metric import report
report(pd.read_csv(ROOT / "futuro" / "oof" / "lgb_future.csv"), "lgb_future (base, 3 semillas)")

## 1 · ¿Qué bloque de información aporta?

**Hipótesis:** casi todo el salto viene del bloque `future` (la siguiente instrucción delata la tarea); `past` y `logic` aportan poco.
Quitar un bloque cada vez dice cuánto vale cada uno. Si quitar uno **no empeora**, sobra (menos ruido, más robustez).

| Variante | Qué se quita | Qué esperamos |
|---|---|---|
| sin `future` | f_* | caída grande (debería acercarse al LightGBM limpio + pasado) |
| sin `past` | p_*, blk_*, gap_* | caída pequeña |
| sin `logic` | logic_* | ruido: con futuro, las reglas a mano deberían sobrar |

In [ ]:
lanzar("feats", [
    ("sin_future", "¿cuánto aporta el futuro?",       dict(feats="base,logic,past")),
    ("sin_past",   "¿cuánto aporta el pasado?",        dict(feats="base,logic,future")),
    ("sin_logic",  "¿sobran las reglas a mano?",       dict(feats="base,past,future")),
])

**Conclusión bloque 1 · casi toda la información está en el futuro; pasado y reglas no aportan.**

| Variante (1 semilla) | CV | Δ vs base (0.900) | peor controlador | |
|---|---|---|---|---|
| sin `logic` | 0.897 | −0.002 | 0.740 | ruido |
| sin `past` | 0.897 | −0.003 | 0.757 | ruido |
| sin `future` | **0.661** | **−0.239** | 0.596 | empeora muchísimo |

- **El bloque `future` es prácticamente todo el modelo:** sin él cae a 0.661, casi al nivel del LightGBM limpio de Adri (0.648).
- **`past` y `logic` no aportan con el futuro presente** (−0.002/−0.003, ruido): lo que dicen ya lo cuenta mejor la siguiente
  instrucción del controlador. No compensa quitarlos tampoco (no mejora), así que la base se queda con los 4 bloques.
- Sin futuro, `base+logic+past` da 0.661 frente a 0.648 de Adri: ~1 punto, que viene sobre todo del pasado (también rompe la
  regla 1). Las `logic_*` son solo del snapshot: Adri ya las tiene en `features.py` de `main` por si quiere probarlas.
- El peor controlador varía bastante entre variantes (0.740–0.757) con 1 semilla: no sacar conclusiones de ahí.

## 2 · Parámetros (los candidatos de Adri, sobre datos con futuro)

**Hipótesis:** con features tan informativas el modelo se apoya en pocas columnas muy fuertes (`f_next_input`…); lo que
le obligue a repartir (menos columnas por árbol) o a ser más prudente puede generalizar mejor a controladores nuevos.

| Variante | Cambios (iguales que en `LGB_adri.ipynb`) | Por qué |
|---|---|---|
| `lento` | `lr=0.03`, `n_estimators=700` | aprender más despacio con más árboles: suele generalizar mejor |
| `regularizado` | `min_child=100`, `colsample=0.5`, `reg_lambda=5` | hojas más grandes, menos columnas por árbol, pesos más pequeños |
| `xendcg` | `objective="rank_xendcg"` | otra pérdida de ranking, más estable con etiquetas ruidosas |

~1 min por variante (1 semilla).

In [ ]:
lanzar("params", [
    ("lento",        "más lento con más árboles (Adri)",                 dict(lr=0.03, n_estimators=700)),
    ("regularizado", "hojas grandes, menos columnas, L2 (Adri)",         dict(min_child=100, colsample=0.5, reg_lambda=5.0)),
    ("xendcg",       "otra pérdida de ranking (Adri)",                   dict(objective="rank_xendcg")),
])

**Conclusión bloque 2 · la CV no se mueve, pero `regularizado` y `lento` mejoran al peor controlador.**

| Variante (1 semilla) | CV | Δ vs base (0.900) | peor controlador | Δ peor | |
|---|---|---|---|---|---|
| lento (`lr=0.03`, 700 árboles) | 0.900 | +0.001 | 0.763 | +0.018 | CV igual, más robusto |
| regularizado (`min_child=100`, `colsample=0.5`, `reg_lambda=5`) | 0.900 | 0.000 | **0.775** | **+0.030** | CV igual, más robusto |
| xendcg | 0.886 | **−0.013** | 0.740 | −0.006 | empeora |

- **En CV media ningún candidato gana:** con el futuro, el modelo base ya está bien ajustado.
- **El peor controlador sí mejora** con `regularizado` (+3 puntos) y `lento` (+1,8). Importa porque el leaderboard privado
  es **un solo controlador** (023) que no está en train: un modelo que no se hunda con ninguno es más seguro. Encaja con la
  hipótesis: obligarle a repartir entre columnas (`colsample=0.5`) evita que dependa solo de `f_next_input`.
- Ojo: con 1 semilla el peor controlador baila (en el bloque 1 iba de 0.740 a 0.757), así que hay que confirmarlo con 3 semillas.
- **`xendcg` se descarta.**

## 3 · Combinar lo que ha funcionado

Pon en `MEJOR` los cambios con veredicto **mejora** de los bloques 1 (p. ej. `feats=...` si sobra un bloque) y 2. Se comprueba con **3 semillas** frente a la base de 3 semillas
(sección 0), porque varios cambios que ayudan por separado no siempre suman.

In [ ]:
MEJOR = dict(
    # bloque 2: misma CV, mejor peor controlador -> se confirma con 3 semillas en la celda siguiente
    min_child=100, colsample=0.5, reg_lambda=5.0,     # regularizado
    lr=0.03, n_estimators=700,                        # lento
)

In [ ]:
# Comprobación con 3 semillas. Se puede saltar: el bloque 4 solo necesita la celda de MEJOR.
cv_comb = probar("comb/" + ",".join(f"{k}={v}" for k, v in MEJOR.items()), "combinación de ganadores", semillas=3, **MEJOR)
print(f"combinación: {cv_comb:.4f}   referencias: {REF}")

**Conclusión bloque 3 · la combinación no mejora: la mejora del peor controlador era ruido de 1 semilla.**

| (3 semillas) | CV | peor controlador | fold 0 | fold 1 | fold 2 | fold 3 | fold 4 |
|---|---|---|---|---|---|---|---|
| **`lgb_future`** (base) | **0.901** | **0.769** | 0.938 | 0.890 | 0.879 | 0.896 | 0.904 |
| `MEJOR` (regularizado + lento) | 0.901 | 0.766 | 0.938 | 0.894 | 0.884 | 0.885 | 0.904 |

- Misma CV y mismo peor controlador: con 3 semillas desaparece la ventaja que se veía con 1 (0.775 vs 0.746).
- Además tarda el doble (700 árboles). **Se queda `lgb_future` como versión final** y el bloque 4 no se lanza.
- Lectura general del barrido: con el futuro, los parámetros de referencia de `EQUIPO.md` ya están bien; el modelo depende de
  la información (bloque 1), no del ajuste.

## 4 · Versión para el stacking (`futuro/`)

Si la combinación mejora a la base, genera la versión definitiva con 3 semillas en el formato común. Nombre nuevo
(`lgb_future_v2`…) para no pisar `lgb_future`. Juanma la combina con `stack.py --futuro`.

In [ ]:
NOMBRE = "lgb_future_v2"
probar(NOMBRE, "versión stacking futuro: " + str(MEJOR), semillas=3, guardar=True, **MEJOR)

**Resultado:** no se lanza; `lgb_future` (CV 0.901, ya en `futuro/oof/` y `futuro/preds/`) es la versión para el stacking de esta rama.

## 5 · Análisis de errores y comparación con la red con futuro

**Pregunta:** ¿fallan el LightGBM y la red en los mismos snapshots? Si no, la combinación de Juanma ganará. Se mira el acierto
por tipo de tarea de cada modelo y en cuántos snapshots acierta uno solo.

In [ ]:
from metric import hits, labels

TIPOS = ["entry_conflict", "exit_conflict", "entry_coord", "exit_coord", "assume", "transfer", "conflict",
         "non_conformance", "qos", "entry_conditions", "exit_conditions", "aircraft_request", "zone", "return_to_route"]


def comparar(modelos=("lgb_future", "nn_future")):
    lab = labels().set_index("sample_id")
    tipos = pd.concat([pd.read_csv(ROOT / "Data" / f"{f}.csv", usecols=["sample_id", "target_index", "task_index"])
                       for f in ("train", "validation")]).query("target_index == 1").set_index("sample_id").task_index
    h = pd.DataFrame({m: hits(pd.read_csv(ROOT / "futuro" / "oof" / f"{m}.csv")) for m in modelos})
    a, b = modelos
    print(f"top-3:  {a} {h[a].mean():.3f}   {b} {h[b].mean():.3f}")
    print(f"aciertan los dos {(h[a] & h[b]).mean():.3f} | solo {a} {(h[a] & ~h[b].astype(bool)).mean():.3f} | "
          f"solo {b} {(~h[a].astype(bool) & h[b]).mean():.3f} | ninguno {((h[a] == 0) & (h[b] == 0)).mean():.3f}")
    h["tipo"] = tipos.reindex(h.index).map(dict(enumerate(TIPOS)))
    t = h.groupby("tipo").agg(muestras=(a, "size"), **{a: (a, "mean"), b: (b, "mean")})
    t["Δ (lgb − red)"] = t[a] - t[b]
    return t.sort_values("muestras", ascending=False).style.format(
        {a: "{:.2f}", b: "{:.2f}", "Δ (lgb − red)": "{:+.2f}"}).background_gradient(subset=["Δ (lgb − red)"], cmap="RdYlGn")


comparar(("lgb_future", "nn_future"))      # cambiar a lgb_future_v2 si se genera

**Conclusión del análisis · los dos modelos fallan casi en los mismos sitios; mezclarlos a partes iguales no suma.**

| | % de snapshots |
|---|---|
| aciertan los dos | 86,6 % |
| solo `lgb_future` | 3,5 % |
| solo `nn_future` | 2,2 % |
| fallan los dos | 7,7 % |

| Tipo correcto | muestras | `lgb_future` | `nn_future` | Δ |
|---|---|---|---|---|
| transfer | 3.527 | 0.98 | 0.98 | +0.00 |
| assume | 3.366 | 0.98 | 0.98 | +0.01 |
| zone_conflict | 2.121 | 0.92 | 0.90 | +0.02 |
| entry_coordination | 1.068 | 0.71 | 0.67 | +0.03 |
| return_to_route | 937 | 0.87 | 0.82 | +0.04 |
| exit_coordination | 745 | 0.57 | **0.59** | **−0.02** |
| exit_conditions | 448 | 0.80 | 0.76 | +0.04 |

- El LightGBM es igual o mejor en casi todos los tipos; la red solo gana en **exit_coordination**.
- Media de puntuaciones normalizadas (z-score dentro del snapshot), según el peso de la red: 0 → **0.901**, 0.3 → 0.900,
  0.5 → 0.897, 1 → 0.887. **Cuanto más peso a la red, peor:** con la información del futuro, los dos aprenden lo mismo.
- Para Juanma: con el futuro, `lgb_future` solo ya es lo mejor. Su meta-modelo (que puede fiarse de la red solo en
  exit_coordination) podría arañar algo, pero no hay que esperar mucho. Contraste con los modelos limpios, donde la red sí
  sumaba (LightGBM 0.648 → mezcla 0.655): sin futuro hay menos información y cada modelo captura cosas distintas.
- (Cifras calculadas con la misma función de la celda anterior; la salida de la celda no quedó guardada en el notebook.)

## 5b · Suavizado temporal de las predicciones

**Hipótesis:** la descripción de los datos dice que los snapshots consecutivos suelen compartir respuesta (el 72 % repite la del
anterior dentro de una racha contigua; rachas de 3 snapshots de mediana). Promediar la probabilidad de cada tarea con la de
la misma tarea en los snapshots vecinos (pesos que decaen con la distancia) debería corregir fallos aislados.

In [ ]:
import numpy as np, features_future as ff

_blk = ff.add_blocks(ff.add_folds(ff.load()))
_blk = _blk[_blk.split != "test"][["sample_id", "row_index", "task_name", "block", "blk_pos"]]


def suavizar(modelo, k=5, decay=0.7):
    """Media ponderada de la probabilidad de la misma tarea en los ±k snapshots contiguos."""
    x = _blk.merge(pd.read_csv(ROOT / "futuro" / "oof" / f"{modelo}.csv"), on=["sample_id", "row_index"])
    x["p"] = np.exp(x.score - x.groupby("sample_id").score.transform("max"))
    x["p"] /= x.groupby("sample_id").p.transform("sum")
    x = x.sort_values(["block", "task_name", "blk_pos"]); g = x.groupby(["block", "task_name"]).p
    num, den = x.p.copy(), pd.Series(1.0, index=x.index)
    for i in range(1, k + 1):
        for sh in (g.shift(i), g.shift(-i)):
            num += decay ** i * sh.fillna(0); den += decay ** i * sh.notna()
    return hits(x[["sample_id", "row_index"]].assign(score=x.p)).mean(), hits(x[["sample_id", "row_index"]].assign(score=num / den)).mean()


pd.DataFrame({m: suavizar(m) for m in ("lgb_future", "nn_future")}, index=["sin suavizar", "suavizado k=5"]).round(4)

**Conclusión 5b:** mejora real pero pequeña (calculado con esta función, k = 2 / 5 / 10):

| | sin suavizar | k=2 | k=5 | k=10 |
|---|---|---|---|---|
| `lgb_future` | 0.901 | 0.906 | 0.906 | **0.907** |
| `nn_future` | 0.887 | 0.895 | **0.898** | 0.898 |

- +0.6 puntos en el LightGBM, +1 en la red: las features de futuro ya contienen buena parte de esa información.
- Para usarlo en test haría falta aplicarlo también a `preds/` (mismas rachas en las sesiones de test). Candidato para la
  versión final de esta rama.

## 6 · Bloques nuevos: `coord` y `clear` (rama `claude/futuro-coord`)

**Por qué:** en la sección 5 lo que queda son sobre todo las **coordinaciones** (entry + exit ≈ 45 % de los fallos) y
`f_next_input` no las ve: coordinar con un sector vecino no es una instrucción al avión. Los dos bloques nuevos miden,
para cada tarea, **cuántos segundos faltan hasta que cambia en la sesión el estado que esa tarea resuelve**:

| Bloque | Archivo | Columnas | Qué cambio futuro mide | Tipos a los que apunta |
|---|---|---|---|---|
| `coord` | `futuro/feat_coord.py` | `sc_*` | coordinación de entrada (`x_isc_*`), de salida (`k_out_*`), assume (`is_assumed` 0→1), transfer (1→0) | entry/exit_coordination, assume, transfer, gemelo assume/entry |
| `clear` | `futuro/feat_clear.py` | `sx_*` | nivel autorizado, rumbo, nivel de salida, fin de zona/conflicto, vuelta a ruta | exit_conditions, zone_conflict, return_to_route, conflictos |

**Hipótesis:** `coord` debería subir sobre todo entry/exit_coordination; `clear` los tipos de instrucciones. Se añaden de
uno en uno sobre la base (los 4 bloques de siempre), y después juntos. 1 semilla, ~1 min por variante.

In [ ]:
lanzar("nuevos", [
    ("+coord",       "cambios futuros de coordinación/assume/transfer",  dict(feats="base,logic,past,future,coord")),
    ("+clear",       "cambios futuros de instrucciones",                 dict(feats="base,logic,past,future,clear")),
    ("+coord+clear", "los dos bloques nuevos",                           dict(feats="base,logic,past,future,coord,clear")),
])

**Conclusión bloque 6:** _(rellenar)_

### 6b · Versión con los bloques que ganen y dónde mejora

Poner en `FEATS_NUEVAS` la combinación ganadora del bloque 6. Se genera con 3 semillas como **`lgb_future_coord`**
(no pisa `lgb_future`) y se compara por tipo de tarea con `lgb_future`: así se ve si mejora justo donde se esperaba
(coordinaciones / instrucciones). ~4 min.

In [ ]:
FEATS_NUEVAS = "base,logic,past,future,coord,clear"     # <- dejar solo los bloques que mejoren
probar("lgb_future_coord", "versión stacking futuro: " + FEATS_NUEVAS, semillas=3, guardar=True, feats=FEATS_NUEVAS)
comparar(("lgb_future_coord", "lgb_future"))

**Conclusión 6b:** _(rellenar)_

## 7 · Siguientes ideas

- Probar aquí las features que está preparando Pau (exit_coordination, return_to_route) cuando estén.
- Si Juanma ve que la combinación `lgb_future` + `nn_future` gana, afinar el peso de cada uno en su `stack.py`.

## Historial completo

In [ ]:
historial()